In [50]:
import sys
sys.path.append('.')  # para que encuentre fetcher.py
import importlib
import fetcher

from fetcher import DataFetcher
from grid_intelligence.params import ENTSOE_API_KEY

importlib.reload(fetcher)

from fetcher import DataFetcher

data_fetcher = DataFetcher(entsoe_api_key=ENTSOE_API_KEY, output_path='raw_data')

# Full fetch
data_fetcher.fetch_full(start='2026-04-18', end='2026-04-20')
print('Full fetch done')

Full fetch from 2026-04-18 to 2026-04-20...
  → ENTSO-E prices...
  → ENTSO-E generation...
  → ENTSO-E load...
  → ENTSO-E wind...
  → Open-Meteo weather...
  → Yahoo Finance TTF gas...
Saved: raw_data/consolidated_full.csv — 193 rows
Full fetch done


In [51]:
print(df.isnull().sum())

price                        0
generation                   2
consumption                  2
wind_onshore                 2
temperature_c                0
humidity_percent             0
cloud_cover_percent          0
shortwave_radiation_wm2      0
wind_speed_ms                0
ttf_gas                    289
dtype: int64


In [52]:
data_fetcher.fetch_delta()
print('Delta done')

Delta fetch from 2026-04-17 to 2026-04-23...
  → ENTSO-E prices...
  → ENTSO-E generation...
  → ENTSO-E load...
  → ENTSO-E wind...
  → Open-Meteo weather...
  → Yahoo Finance TTF gas...
Updated: raw_data/consolidated_full.csv — 577 rows
Delta done


In [53]:
df = pd.read_csv('raw_data/consolidated_full.csv', index_col=0, parse_dates=True)
print(df.index.duplicated().sum())  # sollte 0 sein

0


In [48]:
df.shape

(770, 10)

In [43]:
import pandas as pd

df = pd.read_csv('raw_data/consolidated_full.csv', index_col=0, parse_dates=True)

print(f"Vorher: {df.shape[0]} Zeilen, {df.index.duplicated().sum()} Duplikate")

df = df.sort_index()
df = df[~df.index.duplicated(keep='last')]

print(f"Nachher: {df.shape[0]} Zeilen, {df.index.duplicated().sum()} Duplikate")

df.to_csv('raw_data/consolidated_full.csv')
print("Gespeichert ✓")

Vorher: 674 Zeilen, 193 Duplikate
Nachher: 481 Zeilen, 0 Duplikate
Gespeichert ✓


In [55]:
import pandas as pd

df = pd.read_csv("raw_data/consolidated_full.csv", index_col=0, parse_dates=True)

print("Shape:", df.shape)
print("Von:", df.index.min())
print("Bis:", df.index.max())
print("\nFehlende Werte:")
print(df.isnull().sum())
print("\nErste Zeilen:")
print(df.head())

Shape: (264873, 10)
Von: 2018-09-30 22:00:00+00:00
Bis: 2026-04-21 00:00:00+00:00

Fehlende Werte:
price                      184176
generation                      1
consumption                    17
wind_onshore                    2
temperature_c               86792
humidity_percent            86792
cloud_cover_percent         86796
shortwave_radiation_wm2     86796
wind_speed_ms               86792
ttf_gas                      6122
dtype: int64

Erste Zeilen:
                           price  generation  consumption  wind_onshore  \
datetime_utc                                                              
2018-09-30 22:00:00+00:00  59.53    51434.81          NaN       4292.89   
2018-09-30 22:15:00+00:00    NaN    52085.57          NaN       4239.07   
2018-09-30 22:30:00+00:00    NaN    52345.33          NaN       4208.44   
2018-09-30 22:45:00+00:00    NaN    52902.03          NaN       4153.52   
2018-09-30 23:00:00+00:00  56.10    52799.83     42354.46       4047.26   

       

In [ ]:
df = pd.read_csv('raw_data/consolidated_full.csv', index_col=0, parse_dates=True)
print(df.index.duplicated().sum())  # sollte 0 sein

0


In [48]:
df.shape

(770, 10)

In [43]:
import pandas as pd

df = pd.read_csv('raw_data/consolidated_full.csv', index_col=0, parse_dates=True)

print(f"Vorher: {df.shape[0]} Zeilen, {df.index.duplicated().sum()} Duplikate")

df = df.sort_index()
df = df[~df.index.duplicated(keep='last')]

print(f"Nachher: {df.shape[0]} Zeilen, {df.index.duplicated().sum()} Duplikate")

df.to_csv('raw_data/consolidated_full.csv')
print("Gespeichert ✓")

Vorher: 674 Zeilen, 193 Duplikate
Nachher: 481 Zeilen, 0 Duplikate
Gespeichert ✓


In [ ]:
import pandas as pd

df = pd.read_csv("raw_data/combined_energy_price_clean.csv",
                 sep='\t', low_memory=False)

# Nur DE-LU Sequence 2 PT15M
df_seq2 = df[
    (df['AreaDisplayName'] == 'DE-LU') &
    (df['ResolutionCode'] == 'PT15M')
].copy()

# Index setzen
df_seq2['datetime_utc'] = pd.to_datetime(df_seq2['DateTime(UTC)'], utc=True)
df_seq2 = df_seq2.set_index('datetime_utc')
df_seq2 = df_seq2[['Price[Currency/MWh]']].rename(columns={'Price[Currency/MWh]': 'price'})
df_seq2 = df_seq2.sort_index()

print("Shape:", df_seq2.shape)
print("Von:", df_seq2.index.min())
print("Bis:", df_seq2.index.max())
print("Fehlende Preise:", df_seq2['price'].isna().sum())

Shape: (284156, 1)
Von: 2018-09-30 22:00:00+00:00
Bis: 2026-04-21 21:45:00+00:00
Fehlende Preise: 0


In [ ]:
# consolidated_full laden
df_main = pd.read_csv("raw_data/consolidated_full.csv",
                       index_col=0, parse_dates=True)

# Preis Spalte ersetzen
df_main = df_main.drop(columns=['price'])
df_merged = df_main.join(df_seq2, how='left')

print("Shape:", df_merged.shape)
print("Fehlende Preise:", df_merged['price'].isna().sum())
print(df_merged.head())

Shape: (284169, 10)
Fehlende Preise: 100
                           generation  consumption  wind_onshore  \
datetime_utc                                                       
2018-09-30 22:00:00+00:00    51434.81          NaN       4292.89   
2018-09-30 22:15:00+00:00    52085.57          NaN       4239.07   
2018-09-30 22:30:00+00:00    52345.33          NaN       4208.44   
2018-09-30 22:45:00+00:00    52902.03          NaN       4153.52   
2018-09-30 23:00:00+00:00    52799.83     42354.46       4047.26   

                           temperature_c  humidity_percent  \
datetime_utc                                                 
2018-09-30 22:00:00+00:00            NaN               NaN   
2018-09-30 22:15:00+00:00            NaN               NaN   
2018-09-30 22:30:00+00:00            NaN               NaN   
2018-09-30 22:45:00+00:00            NaN               NaN   
2018-09-30 23:00:00+00:00            NaN               NaN   

                           cloud_cover_percent 

In [9]:
df_merged.to_csv("raw_data/consolidated_seq2.csv")
print("✓ consolidated_seq2.csv gespeichert")
print("Shape:", df_merged.shape)
print("Fehlende Preise:", df_merged['price'].isna().sum())

✓ consolidated_seq2.csv gespeichert
Shape: (284169, 10)
Fehlende Preise: 100


In [ ]:
df = pd.read_csv("raw_data/consolidated_seq2.csv",
                 index_col=0, parse_dates=True)

print("Shape:", df.shape)
print("Von:", df.index.min())
print("Bis:", df.index.max())
print("\nFehlende Werte:")
print(df.isnull().sum())
print("\nErste Zeilen:")
print(df.head())

Shape: (284169, 10)
Von: 2018-09-30 22:00:00+00:00
Bis: 2026-04-21 00:00:00+00:00

Fehlende Werte:
generation                     1
consumption                   17
wind_onshore                   2
temperature_c              86792
humidity_percent           86792
cloud_cover_percent        86796
shortwave_radiation_wm2    86796
wind_speed_ms              86792
ttf_gas                     6503
price                        100
dtype: int64

Erste Zeilen:
                           generation  consumption  wind_onshore  \
datetime_utc                                                       
2018-09-30 22:00:00+00:00    51434.81          NaN       4292.89   
2018-09-30 22:15:00+00:00    52085.57          NaN       4239.07   
2018-09-30 22:30:00+00:00    52345.33          NaN       4208.44   
2018-09-30 22:45:00+00:00    52902.03          NaN       4153.52   
2018-09-30 23:00:00+00:00    52799.83     42354.46       4047.26   

                           temperature_c  humidity_percent  \
date

In [11]:
# Test mit einem kleinen Zeitraum
from entsoe import EntsoePandasClient
import pandas as pd

client = EntsoePandasClient(api_key="3b635cea-3db5-4fab-96ec-80e0b4c1a10a")
start = pd.Timestamp('2024-01-01', tz='Europe/Brussels')
end = pd.Timestamp('2024-01-02', tz='Europe/Brussels')

gen = client.query_generation('10Y1001A1001A82H', start=start, end=end, psr_type=None)
print(gen.columns.tolist())
print(gen.head())

[('Biomass', 'Actual Aggregated'), ('Fossil Brown coal/Lignite', 'Actual Aggregated'), ('Fossil Coal-derived gas', 'Actual Aggregated'), ('Fossil Gas', 'Actual Aggregated'), ('Fossil Hard coal', 'Actual Aggregated'), ('Fossil Oil', 'Actual Aggregated'), ('Geothermal', 'Actual Aggregated'), ('Hydro Pumped Storage', 'Actual Aggregated'), ('Hydro Pumped Storage', 'Actual Consumption'), ('Hydro Run-of-river and poundage', 'Actual Aggregated'), ('Hydro Water Reservoir', 'Actual Aggregated'), ('Other', 'Actual Aggregated'), ('Other renewable', 'Actual Aggregated'), ('Solar', 'Actual Aggregated'), ('Solar', 'Actual Consumption'), ('Waste', 'Actual Aggregated'), ('Wind Offshore', 'Actual Aggregated'), ('Wind Onshore', 'Actual Aggregated'), ('Wind Onshore', 'Actual Consumption')]
                                    Biomass Fossil Brown coal/Lignite  \
                          Actual Aggregated         Actual Aggregated   
2024-01-01 00:00:00+01:00           3949.25                   3352.94   

In [12]:
import pandas as pd
from entsoe import EntsoePandasClient
from dateutil.relativedelta import relativedelta
import time

ENTSOE_API_KEY = "3b635cea-3db5-4fab-96ec-80e0b4c1a10a"
COUNTRY = '10Y1001A1001A82H'
OUTPUT = "raw_data/generation_split.csv"

RENEWABLE = [
    'Biomass', 'Geothermal', 'Hydro Pumped Storage',
    'Hydro Run-of-river and poundage', 'Hydro Water Reservoir',
    'Other renewable', 'Solar', 'Wind Offshore', 'Wind Onshore'
]

NON_RENEWABLE = [
    'Fossil Brown coal/Lignite', 'Fossil Coal-derived gas',
    'Fossil Gas', 'Fossil Hard coal', 'Fossil Oil', 'Waste', 'Other'
]

client = EntsoePandasClient(api_key=ENTSOE_API_KEY)

def fetch_generation_split(start, end):
    gen = client.query_generation(COUNTRY, start=start, end=end, psr_type=None)
    gen = gen.xs('Actual Aggregated', axis=1, level=1)
    gen = gen.resample('15min').mean()
    renewable_cols = [c for c in gen.columns if c in RENEWABLE]
    non_renewable_cols = [c for c in gen.columns if c in NON_RENEWABLE]
    df = pd.DataFrame(index=gen.index)
    df['generation_renewable'] = gen[renewable_cols].sum(axis=1)
    df['generation_non_renewable'] = gen[non_renewable_cols].sum(axis=1)
    return df

# Full fetch
start_dt = pd.Timestamp('2018-10-01', tz='Europe/Brussels')
end_dt   = pd.Timestamp('2026-04-23', tz='Europe/Brussels')
chunks = []
cursor = start_dt

while cursor < end_dt:
    next_cursor = min(cursor + relativedelta(months=3), end_dt)
    print(f"Fetching {cursor.date()} → {next_cursor.date()} ...")
    try:
        df = fetch_generation_split(cursor, next_cursor)
        chunks.append(df)
        print(f"  ✓ {len(df)} rows")
    except Exception as e:
        print(f"  ✗ FEHLER: {e}")
    time.sleep(2)
    cursor = next_cursor

df_gen = pd.concat(chunks).sort_index()
df_gen = df_gen[~df_gen.index.duplicated(keep='last')]
df_gen.to_csv(OUTPUT)
print(f"✓ Gespeichert: {OUTPUT} — {df_gen.shape}")

Fetching 2018-10-01 → 2019-01-01 ...
  ✓ 8836 rows
Fetching 2019-01-01 → 2019-04-01 ...
  ✓ 8636 rows
Fetching 2019-04-01 → 2019-07-01 ...
  ✓ 8736 rows
Fetching 2019-07-01 → 2019-10-01 ...
  ✓ 8832 rows
Fetching 2019-10-01 → 2020-01-01 ...
  ✓ 8836 rows
Fetching 2020-01-01 → 2020-04-01 ...
  ✓ 8732 rows
Fetching 2020-04-01 → 2020-07-01 ...
  ✓ 8736 rows
Fetching 2020-07-01 → 2020-10-01 ...
  ✓ 8832 rows
Fetching 2020-10-01 → 2021-01-01 ...
  ✓ 8836 rows
Fetching 2021-01-01 → 2021-04-01 ...
  ✓ 8636 rows
Fetching 2021-04-01 → 2021-07-01 ...
  ✓ 8736 rows
Fetching 2021-07-01 → 2021-10-01 ...
  ✓ 8832 rows
Fetching 2021-10-01 → 2022-01-01 ...
  ✓ 8836 rows
Fetching 2022-01-01 → 2022-04-01 ...
  ✓ 8636 rows
Fetching 2022-04-01 → 2022-07-01 ...
  ✓ 8736 rows
Fetching 2022-07-01 → 2022-10-01 ...
  ✓ 8832 rows
Fetching 2022-10-01 → 2023-01-01 ...
  ✓ 8836 rows
Fetching 2023-01-01 → 2023-04-01 ...
  ✓ 8636 rows
Fetching 2023-04-01 → 2023-07-01 ...
  ✓ 8736 rows
Fetching 2023-07-01 → 2023-10-0

In [1]:
import pandas as pd

df_main = pd.read_csv("raw_data/consolidated_full.csv", index_col=0, parse_dates=True)
df_gen = pd.read_csv("raw_data/generation_split.csv", index_col=0, parse_dates=True)

df_merged = df_main.join(df_gen, how='left')

print("Shape:", df_merged.shape)
print("Fehlende Renewable:", df_merged['generation_renewable'].isna().sum())

df_merged.to_csv("raw_data/consolidated_full.csv")
print("✓ consolidated_full.csv updated")

Shape: (265065, 12)
Fehlende Renewable: 9
✓ consolidated_full.csv updated


In [2]:
import pandas as pd

df_main = pd.read_csv("raw_data/consolidated_full.csv", index_col=0, parse_dates=True)
df_gen = pd.read_csv("raw_data/generation_split.csv", index_col=0, parse_dates=True)

print("consolidated_full:", df_main.shape)
print("generation_split:", df_gen.shape)

df_merged = df_main.join(df_gen, how='left')

print("\nShape nach merge:", df_merged.shape)
print("Fehlende generation_renewable:", df_merged['generation_renewable'].isna().sum())
print("Fehlende generation_non_renewable:", df_merged['generation_non_renewable'].isna().sum())

consolidated_full: (265065, 12)
generation_split: (265056, 2)


ValueError: columns overlap but no suffix specified: Index(['generation_renewable', 'generation_non_renewable'], dtype='object')

In [3]:
print(df_main.columns.tolist())

['generation', 'consumption', 'wind_onshore', 'temperature_c', 'humidity_percent', 'cloud_cover_percent', 'shortwave_radiation_wm2', 'wind_speed_ms', 'ttf_gas', 'price', 'generation_renewable', 'generation_non_renewable']


In [4]:
print(df_main[['generation_renewable', 'generation_non_renewable']].isna().sum())
print(df_main[['generation_renewable', 'generation_non_renewable']].head(10))

generation_renewable        9
generation_non_renewable    9
dtype: int64
                           generation_renewable  generation_non_renewable
datetime_utc                                                             
2018-09-30 22:00:00+00:00              11863.35                  28945.23
2018-09-30 22:15:00+00:00              11809.11                  29001.10
2018-09-30 22:30:00+00:00              11632.71                  29136.32
2018-09-30 22:45:00+00:00              11937.48                  28929.82
2018-09-30 23:00:00+00:00              12005.45                  28880.65
2018-09-30 23:15:00+00:00              11810.22                  28905.60
2018-09-30 23:30:00+00:00              11715.66                  28744.03
2018-09-30 23:45:00+00:00              11736.31                  28566.58
2018-10-01 00:00:00+00:00              11894.87                  28465.91
2018-10-01 00:15:00+00:00              11880.10                  28616.98


In [6]:
import pandas as pd

df = pd.read_csv("raw_data/consolidated_full.csv", index_col=0, parse_dates=True)
print("Shape:", df.shape)
print("Letztes Datum:", df.index.max())
print("Erstes Datum:", df.index.min())

Shape: (265065, 12)
Letztes Datum: 2026-04-23 00:00:00+00:00
Erstes Datum: 2018-09-30 22:00:00+00:00
